# loss-item-scalar-extract — ex1: distinguish .item() from .detach().cpu() in a training-loop logger

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `loss-item-scalar-extract`. Running the final beacon cell reports progress against the `PyTorch: loss.item() scalar extract` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: loss.item() scalar extract` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`loss-item-scalar-extract`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "loss-item-scalar-extract"
DD_SUBTOPIC = "PyTorch: loss.item() scalar extract"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `loss.item()` scalar extraction — quick refresher

Inside a training loop you frequently need the loss as a Python float — to print, to log, to compare against a threshold. **`.item()` is the only correct way** for 0-D (scalar) tensors:

```python
loss = F.cross_entropy(logits, labels)   # 0-D tensor on GPU
scalar = loss.item()                     # Python float, autograd-detached
wandb.log({'loss': scalar})
```

**Three patterns, three jobs.**
1. **`.item()`** — extracts ONE scalar. Synchronizes (forces CUDA → CPU stall). Use for logging / branching. Errors on multi-element tensors.
2. **`.detach().cpu()`** — keeps the tensor structure, just removes autograd + moves to CPU. Use when you want to **buffer** many step values (append to a list, then `torch.stack` later) without holding the compute graph hostage.
3. **`float(loss)`** — works for 0-D tensors via the `__float__` dunder, but PyTorch deprecated this for >=0-D ambiguity. Use `.item()` instead.

**Why `.item()` synchronizes.** It must wait for the kernel that produced the loss to finish before reading the value back. Calling `.item()` every step is fine; calling it on intermediate activations inside the inner loop tanks throughput.

**The autograd half.** `.item()` automatically detaches — the Python float has no `.grad_fn`. This is what makes it safe to store: `loss_history.append(loss)` would pin the compute graph forever; `loss_history.append(loss.item())` is a one-time cost.

### Exercise 1 — distinguish .item() from .detach().cpu() in a training-loop logger

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Use `.item()` to log a scalar float and `.detach().clone()` to buffer many step values, then assert that `.item()` returns a Python float (not a Tensor) and breaks the autograd graph.
> Keywords: loss, item, detach, logging, graph-pinning
> ```

**KCs targeted:** `loss-item-returns-python-float`, `detach-clone-for-buffer`

Implement `ex1_train_log(steps)` — a fake training loop that logs the loss in TWO ways and returns BOTH:

1. `steps`: int, number of training iterations to simulate.
2. Run a synthetic training loop: set `x = t.randn(steps, requires_grad=False)` as the per-step input. The 'loss' for step `i` is `loss_i = (x[i] ** 2 + 1.0).requires_grad_(True)` — a 0-D scalar tensor with a real autograd graph attached (so `.grad_fn` is not None).
3. For each step, populate TWO logs:
   - `floats_log[i] = loss_i.item()` — Python float for wandb-style logging.
   - `tensors_log[i] = loss_i.detach().clone()` — detached 0-D tensor for in-memory buffering.
4. After the loop, return a dict:
   - `'floats_log'`: a Python `list[float]` of length `steps`.
   - `'tensors_log'`: a tensor of shape `(steps,)` built via `t.stack(tensors_log)`.
   - `'sample_float_type'`: `type(floats_log[0]).__name__` — must be `'float'`.
   - `'sample_tensor_type'`: `type(tensors_log[0]).__name__` — must be `'Tensor'`.
   - `'sample_tensor_grad_fn'`: `tensors_log[0].grad_fn` — must be `None` (detached).

**Why both patterns exist.** `.item()` synchronizes and extracts ONE value — perfect for wandb / tqdm / print. `.detach().clone()` keeps the tensor shape, breaks the autograd graph (so the compute history can be freed), and is the right choice for buffering many step values for later analysis.

**The sabotage trap.** Do NOT do `tensors_log[i] = loss_i` (without detach). That keeps the autograd graph alive across all `steps` — memory grows linearly, and `tensors_log[0].grad_fn` would be non-None. The test asserts grad_fn is None to catch this exact bug.

In [ ]:
def ex1_train_log(steps: int) -> dict:
    x = t.randn(steps, requires_grad=False)
    floats_log = []
    tensors_log = []
    for i in range(steps):
        loss_i = (x[i] ** 2 + 1.0).requires_grad_(True)
        floats_log.append(loss_i.item())              # Python float for logging
        tensors_log.append(loss_i.detach().clone())   # detached tensor for buffering
    stacked = t.stack(tensors_log)
    return {
        'floats_log': floats_log,
        'tensors_log': stacked,
        'sample_float_type': type(floats_log[0]).__name__,
        'sample_tensor_type': type(tensors_log[0]).__name__,
        'sample_tensor_grad_fn': tensors_log[0].grad_fn,
    }


<details><summary>Solution</summary>

```python
def ex1_train_log(steps: int) -> dict:
    x = t.randn(steps, requires_grad=False)
    floats_log = []
    tensors_log = []
    for i in range(steps):
        loss_i = (x[i] ** 2 + 1.0).requires_grad_(True)
        floats_log.append(loss_i.item())              # Python float for logging
        tensors_log.append(loss_i.detach().clone())   # detached tensor for buffering
    stacked = t.stack(tensors_log)
    return {
        'floats_log': floats_log,
        'tensors_log': stacked,
        'sample_float_type': type(floats_log[0]).__name__,
        'sample_tensor_type': type(tensors_log[0]).__name__,
        'sample_tensor_grad_fn': tensors_log[0].grad_fn,
    }
```

**The graph-pinning bug.** Every operation on a `requires_grad=True` tensor records itself in the autograd graph for later `.backward()`. If you store `loss_i` directly (without `.detach()`), the graph that produced it stays rooted in your list — memory grows with `steps`, and subsequent `.backward()` calls re-traverse it. The `.detach()` call strips the `.grad_fn` reference, letting Python GC reclaim the graph nodes as soon as the next step's `loss_i` goes out of scope.

**Why `.clone()` after `.detach()`.** `.detach()` returns a VIEW of the same storage with `requires_grad=False`. The `.clone()` makes a copy, which is important if the buffer outlives the source tensor — without the clone, mutations to the original (e.g. `loss_i.zero_()`) would corrupt every logged value. For 0-D scalars from `.item()`-flavored logging this is paranoia; for buffering intermediate activations it's essential.

**The `.item()` synchronization cost.** On CUDA, `.item()` forces a host-device sync — it must wait for the kernel producing the value to finish before reading. One `.item()` per training step is fine (you're already waiting on the backward pass). One `.item()` per layer or per attention head tanks throughput — the kernel pipeline drains every call.

**Why `float(loss)` is deprecated.** It worked for 0-D tensors via the `__float__` dunder, but PyTorch deprecated it for ambiguity — `float(some_1d_tensor)` would just take the first element. `.item()` raises a clear error on multi-element tensors, which is the safer behavior.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()